<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11.2-energy-optimization/Ex11.2_pinn_cheapest_lap_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_11.2 — The Cheapest Lap

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L11.2 · Dynamics, Energy and Efficient Driving**

A JetRacer drives a closed course. Which speed, at every point of the lap,
makes the lap cost least - counting the energy from the pack and the time it
takes? Too slow, and the electronics are paid for longer. Too fast, and the
energy thrown away before every corner has to be bought again. You find the
profile two ways - a classical optimiser on a stepped lap, and a network with
the grip limit built in - and compare them on cost and on time. Then a
physics-informed fit finds the rolling coefficient from a coast-down.

This notebook runs on Colab or a laptop. **The car's numbers in it are
placeholders**: the set's other notebook, `Ex11.2_on_the_car.ipynb`, measures
them on your own car and drives the scored laps.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and the classical optimiser on a step for an agreed 1 % | — |
| **5** | a network for the speed profile, and the comparison | the bounded profile, the lap cost |
| **6** | the inverse problem: the rolling coefficient from a coast-down | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About twenty seconds on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11.2-energy-optimization/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex11.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The course is 18 m round: four straights and three corners, of radius 1.11,
0.71 and 1.67 m. The car may not go faster through a corner than its tyres
hold, and nowhere faster than its top speed. Wanted: the speed at every point
of the lap that makes

$$J = E + w\,T$$

smallest, where $E$ is the energy drawn from the pack over the lap, $T$ the
lap time, and $w$ = 5 J/s says what a second is worth.

## 2 · The data

| | |
|---|---|
| car | 1.05 kg, tyre friction $\mu$ = 0.65, rolling coefficient $c_{rr}$ = 0.02 |
| motor | traction constant $K$ = 1.0 N/A, armature resistance $R_a$ = 1.5 Ω |
| electronics | $P_{\mathrm{hotel}}$ = 7 W: the Nano, the camera, the display |
| limits | top speed 3.0 m/s, stall below 0.3 m/s |
| course | 18 m: straights and three corners |
| cost | $w$ = 5 J/s |

**Every number of the car is a placeholder of the right order, not a
measurement.** The on-car notebook measures them. Until you put your own
values into `problem.py`, the answer below is the answer for this imagined
car.

## 3 · The physics

**The force** the motor must supply is rolling resistance, drag and the
force that accelerates the car (L11.2's *Brushed DC Motor*):

$$F = c_{rr}\,m g + \tfrac12 \rho\,C_d A\,v^2 + m\,v\,\frac{dv}{ds}$$

**The power** from the pack is the heat in the winding, the traction power
and the electronics. There is **no regeneration**: braking draws no current
and returns none.

$$i = \max\!\Big(\frac{F}{K},\,0\Big), \qquad P = i^2 R_a + K\,i\,v + P_{\mathrm{hotel}}$$

**The grip limit** (L11.2's *What the Car Physically Cannot Do*): in a corner
of curvature $\kappa$ the speed cannot pass $\sqrt{\mu g/\kappa}$.

Over a lap, $E = \sum P\,\Delta t$ and $T = \sum \Delta t$ with
$\Delta t = \Delta s / v$. The cell below prints the data, draws the lap
unrolled along its length, and draws the ceiling the speed has to stay under.

In [ ]:
pb.describe_problem()
pb.draw_course(); plt.show()                                 # the lap, unrolled: straights and corners
s_fine, curv_fine = pb.course(0.05)
fig, ax = plt.subplots(figsize=(9, 3.0))
ax.plot(s_fine, pb.ceiling(curv_fine), "k--", lw=1.5, label="the ceiling: grip in the corners, top speed on the straights")
ax.axhline(pb.V_FLOOR, color="0.6", lw=1, label="stall")
ax.set_xlabel("arc length round the lap  [m]"); ax.set_ylabel("speed  [m/s]"); ax.set_ylim(0, 3.4)
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

**What you should see.** The placeholders of section 2, grip limits of 2.66,
2.13 and 3.26 m/s in the three corners (the third is above the top speed, so
it does not bind), and the cost of a metre driven steadily: 24.3 J at 0.5 m/s,
8.3 J at 1.5 m/s, 4.3 J at 3.0 m/s. With 7 W of electronics and 5 J for every
second, **time is the expensive part**: on a straight, faster is cheaper. The
ceiling drops in two of the corners.

---

## 4 · The reference, and the classical optimiser

**The classical way** is to cut the lap into steps, make the speed in each
step one unknown, bound each between the stall speed and the ceiling, and hand
the cost to a quasi-Newton optimiser with bounds, L-BFGS-B.
`pb.optimise(ds)` does that.

**The reference.** There is no formula for the best lap, so the reference is
the same optimiser on a much finer step. The cell below halves the step four
times.

**The agreed accuracy is 1 %** of the cost $J$.

In [ ]:
runs = {}
for ds in (1.0, 0.5, 0.25, 0.125, 0.0625):
    t0 = time.perf_counter(); runs[ds] = pb.optimise(ds); sec = time.perf_counter() - t0
    runs[ds] += (sec,)
    _, v, e, t, j, _ = runs[ds]
    print(f"  step {ds:6.4f} m, {len(v):3d} unknowns: energy {e:.2f} J, lap {t:.2f} s, cost {j:.2f} J, {sec:.2f} s")
J_REF = runs[0.0625][4]                                       # the reference: the finest step
for ds in (1.0, 0.5, 0.25, 0.125):
    off = 100 * abs(runs[ds][4] / J_REF - 1)
    print(f"  step {ds:5.3f} m against the reference: {off:.2f} %" + ("   <- meets 1 %" if off <= 1.0 else ""))
DS = 0.25                                                     # the coarsest step that meets it
s, v_classic, E_classic, T_classic, J_classic, t_classic = runs[DS]

**What you should see.** The cost settling as the step halves - 84.03, 87.45,
88.03, 88.33 and 88.50 J - and **a step of 0.25 m, 72 unknowns, the first
within 1 %** of the finest, in well under a second. The best lap takes about
6.5 s and 55 J: the car runs at the ceiling almost everywhere.

---

## 5 · A network for the speed profile, and the comparison

**The profile as a function.** A network maps the position on the lap to a
speed, so the answer is a function of arc length and not a list of numbers.
Its inputs are six harmonics of the lap, $\sin$ and $\cos$ of
$2\pi k s/L$, so that the end of the lap joins its start. It is three layers
of 32 tanh neurons.

**The grip limit is built in.** The last layer is

$$v(s) = v_{\mathrm{floor}} + \sigma\big(\mathcal{N}(s)\big)\,\big(v_{\mathrm{ceil}}(s) - v_{\mathrm{floor}}\big)$$

with $\sigma$ the logistic function, which lies between 0 and 1. Whatever the
network does, the speed is between the stall speed and the ceiling: a limit
the optimiser cannot trade away (L11.2's *Two Ways to Enforce a Constraint*).

**The loss is the lap cost itself.** There is no data and no residual: the
physics is the cost, written in torch so that autograd can differentiate it.

**TODO 1** is the bounded profile; **TODO 2** the lap cost.

In [ ]:
HARMONICS = 6
def features(s):
    # sin and cos of the first six harmonics of the lap: the profile then closes on itself
    return np.column_stack([f(2 * np.pi * k * s / pb.LAP_LENGTH) for k in range(1, HARMONICS + 1) for f in (np.sin, np.cos)])

class Profile(torch.nn.Module):                               # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2 * HARMONICS, n_hidden=32, n_layers=3)   # dense tanh network: twelve features in
    def forward(self, x, top):
        return pb.V_FLOOR + torch.sigmoid(self.net(x)) * (top - pb.V_FLOOR)   # between the stall speed and the ceiling

In [ ]:
def lap_cost(v):
    # the cost J = E + w T of a lap at speeds v (a tensor, one speed per step of DS), differentiable
    dv_ds = (torch.roll(v, -1) - torch.roll(v, 1)) / (2 * DS)  # dv/ds round the closed lap
    force = pb.C_RR * pb.MASS * pb.G + 0.5 * pb.RHO_AIR * pb.CD_A * v ** 2 + pb.MASS * v * dv_ds
    current = torch.clamp(force / pb.K_TRACTION, min=0.0)      # no regeneration: braking draws nothing
    power = current ** 2 * pb.R_ARM + pb.K_TRACTION * current * v + pb.P_HOTEL
    dt = DS / v                                                # the time spent in each step
    return (power * dt).sum() + pb.W_TIME * dt.sum()           # energy, plus what the time is worth

In [ ]:
_, curv = pb.course(DS)
X = to_tensor(features(s))                                    # the twelve features at each step
TOP = to_tensor(pb.ceiling(curv).reshape(-1, 1))              # the ceiling at each step

model = Profile().to(DEVICE)                                  # three layers of 32
print(f"{parameter_count(model)} parameters for a profile of {len(s)} steps")   # the network is larger than the list it replaces
loss_fn = lambda: lap_cost(model(X, TOP).squeeze(1)) / 100.0  # the loss: the lap cost, in hundreds of joules
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=100, lr=5e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
with torch.no_grad():  # no gradient tracking: evaluation only
    v_net = to_numpy(model(X, TOP)).ravel()                   # the network's speed at each step
E_net, T_net, J_net = (float(a) for a in pb.lap_cost(v_net, DS))
over = float(np.max(v_net - pb.ceiling(curv)))                # how far above the ceiling, anywhere

print(f"{'':36s}{'energy':>10s}{'lap time':>10s}{'cost J':>10s}{'against the reference':>23s}{'computed in':>13s}")
print(f"  {'reference, step 0.0625 m':34s}{runs[0.0625][2]:8.2f} J{runs[0.0625][3]:8.2f} s{J_REF:8.2f} J{'-':>23s}{runs[0.0625][5]:11.2f} s")
print(f"  {'classical optimiser, step 0.25 m':34s}{E_classic:8.2f} J{T_classic:8.2f} s{J_classic:8.2f} J{100 * (J_classic / J_REF - 1):21.2f} %{t_classic:11.2f} s")
print(f"  {'network, grip limit built in':34s}{E_net:8.2f} J{T_net:8.2f} s{J_net:8.2f} J{100 * (J_net / J_REF - 1):21.2f} %{train_time:11.2f} s")
print(f"  the network against the classical optimiser on the same step: {100 * (J_net / J_classic - 1):+.2f} %")
print(f"  the network above the ceiling, at worst: {max(over, 0.0):.2f} m/s")

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(s_fine, pb.ceiling(curv_fine), "k--", lw=1.2, label="ceiling")
ax.plot(s, v_classic, lw=2.5, label="classical optimiser")
ax.plot(s, v_net, ":", lw=2.5, label="network")
ax.set_xlabel("arc length round the lap  [m]"); ax.set_ylabel("speed  [m/s]"); ax.set_ylim(0, 3.4)
ax.legend(frameon=False, loc="lower right"); plt.tight_layout()
plt.savefig(cc.output_path("Ex11.2_report.png"), dpi=150); plt.show()

**What you should see.** Two profiles almost on top of each other: at the
ceiling on the straights, slowing into the two tight corners and speeding up
out of them. The classical optimiser's lap costs 88.03 J and the network's
about 88.08 J, within about 0.05 % of each other and both within 1 % of the
reference. The network is never above the ceiling, by construction. Both are
computed in a few seconds or less.

---

## 6 · The inverse problem: the rolling coefficient from a coast-down

The lap above is only as good as the car's numbers. The cleanest way to
measure one of them is a **coast-down**: bring the car to speed, cut the
drive, and log how the speed falls. With no current every electrical term is
zero, and what is left is

$$m\,\frac{dv}{dt} = -\,c_{rr}\,m g - \tfrac12 \rho\,C_d A\,v^2$$

It is L11.2's *Fitting the Model to a Drive*. The log here is **synthetic**:
that equation at $c_{rr}$ = 0.02, sampled every 50 ms for 6 s, with 0.03 m/s
of noise. The car has no encoder, so on the real car the speed has to come
from lap timing or from the camera.

Two ways again. The classical one integrates the equation for a trial
$c_{rr}$ and adjusts it by least squares. The physics-informed one trains a
small network for $v(t)$ **and** the number $c_{rr} = e^{\beta}$ together: the
loss is the residual of the equation plus the misfit to the log.

In [ ]:
from scipy.optimize import least_squares
t_cd, v_cd = pb.coast_down()                                  # 121 readings over 6 s, with noise
t0 = time.perf_counter()
fit = least_squares(lambda p: pb.coast_down_model(t_cd, p[0]) - v_cd, [0.05])   # the classical fit, from 0.05
t_ls = time.perf_counter() - t0

T_CD, V0 = 6.0, 2.5                                           # the scales: the window, and the starting speed
tau = to_tensor((t_cd / T_CD).reshape(-1, 1), requires_grad=True)   # scaled time, differentiable
v_obs = to_tensor((v_cd / V0).reshape(-1, 1))                 # the readings, scaled

class CoastDown(torch.nn.Module):                             # a network for the speed, and one more trainable number
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=1, n_hidden=16, n_layers=2)       # a small dense tanh network: time in
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(0.05))))   # c_rr = exp(beta), starts at 0.05
    def forward(self, tau):
        return 1.0 + tau * self.net(tau)                      # the speed over V0: exactly 1 at the start
cd = CoastDown().to(DEVICE)                                   # the network and beta, on the device
def coast_loss():
    u = cd(tau)                                               # the network's scaled speed
    du = grad(u, tau)                                         # its time derivative, by autograd
    c = torch.exp(cd.beta)                                    # the rolling coefficient being identified
    res = du * V0 / T_CD + c * pb.G + 0.5 * pb.RHO_AIR * pb.CD_A * (V0 * u) ** 2 / pb.MASS   # the equation, per kg
    return res.pow(2).mean() + 10.0 * (u - v_obs).pow(2).mean()   # the residual, and the misfit to the log
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(cd, coast_loss, adam_steps=500, lbfgs_steps=50, lr=5e-3, report_every=0)
t_pi = time.perf_counter() - t0
c_ls, c_pi = float(fit.x[0]), float(torch.exp(cd.beta).detach())   # the two answers
print(f"rolling coefficient: least squares {c_ls:.4f} ({t_ls:.2f} s), physics-informed fit {c_pi:.4f} ({t_pi:.2f} s), true {pb.C_RR:.4f}")

**What you should see.** Both fits return a rolling coefficient of about
0.0200, the value the log was made with, from a start at 0.05.

---

## 7 · What the notebook says

* **With these numbers, time is the expensive part.** The electronics draw
  7 W whether the car moves or not, and a second is worth 5 J more. So the
  cheapest lap is close to the fastest one the tyres allow, and what the
  optimiser decides is how to enter and leave the corners.
* **No regeneration changes the corners.** The kinetic energy shed before a
  corner is not returned, so every corner costs the energy to speed up again.
* **The two methods agree to a fraction of a per cent.** For 72 unknowns and
  one lap the classical optimiser is the right tool. The network is a function
  and not a list, and its grip limit cannot be violated - which matters when
  the profile becomes a policy that reacts to what the camera sees.
* **A built-in limit is not a penalty.** The network never exceeded the
  ceiling. A penalty would have let it, a little, wherever that paid.
* **The inverse problem needed one more trainable number.** The same loop,
  with the rolling coefficient trained beside the weights, returned it from a
  noisy log - as the least-squares fit did, for a model this small.
* **None of the joules above are your car's.** Every parameter is a
  placeholder until the on-car notebook has measured it, and the model is
  believed only after it has predicted a lap that was then measured.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex11.2_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L11.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The notebook's car is a point moving along the course. Which of its numbers would you have to measure on your own car before using the result, and which did section 6 show how to measure? (-> L11.2 Q4)": """
""",
    "How was the grip limit put into the network, and what would have been different if it had been a penalty in the loss? (-> L11.2 Q1)": """
""",
    "Where does the energy of the best lap go, and why was the cheapest lap here close to the fastest one? Say which of the placeholders would have to change for a slower lap to win. (-> L11.2 Q1)": """
""",
    "The steering never appears in this notebook. What does the bicycle model add that a speed profile along a fixed line leaves out? (-> L11.2 Q2)": """
""",
    "To conclude: the classical optimiser and the network agreed. Name one change to the problem that would make the network the better tool, and say what you would measure on the car to check the lap it predicts. (-> L11.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | energy | lap time | cost J | against the reference | computed in |", "|---|---|---|---|---|---|",
            f"| reference, step 0.0625 m | {runs[0.0625][2]:.2f} J | {runs[0.0625][3]:.2f} s | {J_REF:.2f} J | - | {runs[0.0625][5]:.2f} s |",
            f"| classical optimiser, step 0.25 m | {E_classic:.2f} J | {T_classic:.2f} s | {J_classic:.2f} J | {100 * (J_classic / J_REF - 1):.2f} % | {t_classic:.2f} s |",
            f"| network, grip limit built in | {E_net:.2f} J | {T_net:.2f} s | {J_net:.2f} J | {100 * (J_net / J_REF - 1):.2f} % | {train_time:.2f} s |",
            "", f"Coast-down: rolling coefficient {c_ls:.4f} by least squares, {c_pi:.4f} by the physics-informed fit, true {pb.C_RR:.4f}",
            "", "The car's parameters are placeholders, not measurements."]
    out = ["# Ex_11.2 — The Cheapest Lap", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex11.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex11.2_report.md into Ex11.2_report.pdf, with any figure
# saved as Ex11.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex11.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex11.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex11.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex11.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP11.2A · The energy-optimal lap, with the battery

**The problem.** The set found the energy-optimal lap time from the car's measured parameters.
Here the lap is a fixed track, the grip limits the turns, the motor's copper
loss makes hard acceleration expensive, and the pack's voltage sags with the
current it delivers and falls as it discharges.

| | |
|---|---|
| track | the stadium of MP11.1A: 7.20 m, turns of 0.7 m radius |
| motor | the on-car notebook's model: $F$ = $m v\,dv/ds$ + rolling + drag; power $(F/K)^2 R_a + F v$, no regeneration |
| car | this set's placeholders - 1.05 kg, $\mu$ 0.65, $c_{rr}$ 0.02, $K$ 1.0 N/A, $R_a$ 1.5 Ω - and a top speed of 3.0 m/s; hotel load 7 W |
| pack | 2S2P, 8.4 V full (vendor); 2.5 Ah per cell, 50 mΩ, falling linearly to 6.4 V empty |

Every value but the pack's 8.4 V is assumed until measured on the car; the
turns cap the speed at 2.11 m/s.

**What you build.** The network of section 5 extended with the pack: the speed profile over arc
length, the grip limit built in as a layer, the energy drawn through the
pack's resistance. You find the energy for each lap time and the cheapest
lap.

**The ground truth you get.** `MP11.2A_truth.npz`, from `tools/miniprojects/ex112_truth.py`: the optimal
profile by direct collocation (scipy's SLSQP on the speeds at 50, 100 and 200
nodes), for lap times from 3.3 to 8 s, full and at 10 % charge.

**Worked example.** 

| lap time | energy, full pack | at 10 % charge |
|---|---|---|
| 3.3 s | 27.23 J | 27.34 J |
| 3.4 s | **26.16 J** | 26.25 J |
| 4.0 s | 30.28 J | 30.39 J |
| 6.0 s | 44.29 J | 44.43 J |
| 8.0 s | 58.41 J | 58.60 J |

The cheapest lap is 3.4 s - constant speed at the turns' limit. Slower, the
hotel load runs longer; faster, the car must accelerate and brake on the
straights and the copper loss grows with the square of the force. The pack's
sag adds under 1 %. At 3.4 s the energy is the same on 50 to 400 nodes; at 3.3
s it still moves by 1 % (27.12, 27.23, 27.42 J).

**Requirements.**

* The energy within **2 %** at every lap time from 3.4 to 8 s.
* The cheapest lap time within **0.1 s**.
* The whole energy curve in under **1 s**, against the collocation's minutes.

**What you hand in.** Your energy curve against the ground truth's, the optimal profile at the
cheapest lap, what the pack's sag costs, and one change to the problem that
would make the network the better tool.

### MP11.2B · What the model misses

**The problem.** The set identified four lumped parameters and said what the data could not
separate. Real cars also lose energy in ways a four-parameter model does not
have. Here a network learns that missing part from recorded runs, alongside
the known physics.

| | |
|---|---|
| model | $m\,dv/dt = -(m g c_{rr} + \tfrac12\rho C_dA v^2) - f_{\text{extra}}(v)$, the four parameters known in form |
| hidden | an extra friction of the form $a\,e^{-v/v_0}$ - strong at low speed, fading above about 0.4 m/s |
| records | coast-downs from 1, 2 and 3 m/s, speed every 20 ms for 6 s, noise 0.02 m/s |

The car has no encoder, so the runs are simulated; the parameters are
this set's placeholders, assumed until measured.

**Why a plain PINN is not enough.** A network that is free to explain everything will also explain the part the
physics already knows, and then the parameters mean nothing. Keeping the
learned part small, and in the right place, is the problem.

**What you build.** A neural ODE: the model of this set with its parameters trainable, plus a
small network for what it leaves out, fitted to the coast-downs. You show that
the parameters stay meaningful.

**The ground truth you get.** `MP11.2B_readings.npz` to fit and `MP11.2B_truth.npz` with the true
parameters and extra loss, from the same script (integrated at 1e-10).

**Worked example.** With the extra friction the car coasting from 1 m/s stops at 3.10 s; the
four-parameter model misses its speed by up to 0.081 m/s from 1 m/s, 0.068
from 2 m/s and only 0.006 from 3 m/s - the missing loss lives at low speed,
where a fast coast-down never looks.

**Requirements.**

* The learned extra loss within **20 %** of the true one from 0.1 to 1 m/s.
* The four parameters within **10 %** - not absorbed by the network.
* The speed within **0.02 m/s** over every run.

**What you hand in.** The recovered parameters and the learned extra loss against the true ones, the
lap energy each predicts, and how you kept the network from absorbing the
physics.